# 022-adaptive-gate checks

# Imports

In [ ]:
import json
import os

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import numpy as np
import pandas as pd

from uavloc.data.drone_frames import DroneFrame, load_frames
from uavloc.data.multidate import load_date
from uavloc.features.factory import correlator_config
from uavloc.features.segmentation import SegmentationFeatures, load_segmenter
from uavloc.geo import LatLon
from uavloc.matching.benchmark import benchmark_draws
from uavloc.matching.correlator import CorrelatorConfig, PositionCorrelator, SearchSurface
from uavloc.matching.risk import feature_table, gate_features
from uavloc.track import REPO_ROOT, experiment_dir

# Params

In [ ]:
PLACE = "bushcha-50p2422-25p8305"
REFERENCE_RECORDS = "007-search-prior-and-gate/artifacts/records-007.8.csv"
TAUS = (0.25, 0.5, 1.0, 2.0)
NOISE_M = 10.0
N_DIRECTIONS = 8
SEED = 42
DEVICE = "mps"
FRAMES_DATASET = "village-crossdate-frames"
SEG_CHECKPOINT = "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt"
MATCH_GSD_M = 0.385
MAP_DATE = "newest"
HEATMAP_CELLS = 61
EDGE_CELLS = 70
DRONE_CELLS = 10
BUMP_SIGMA_CELLS = 8.0
LINE_SIGMA_CELLS = 3.0
MAX_BUMP_AXIS_RATIO = 1.3
MIN_LINE_AXIS_RATIO = 3.0
MAX_RIDGE_TAU = 0.5   # a ridge washes out once tau is large against its z contrast
MAX_FEATURE_DIFF = 1e-4   # metres; the features add LOG_EPS = 1e-6 inside the log
RUN_CONFIG = {"method": "segmentation", "matching": "weighted-ncc", "class_weights": (0.0, 1.0, 1.0, 0.5, 0.0, 0.5),
              "window": "rect", "search_scale": 1.7, "search_radius_m": None, "prior_sigma_m": None, "gate": "none",
              "fov_w_deg": 60.0, "sat_feature_crop_pct": 0.03, "drone_feature_crop_pct": 0.0}

# Synthetic

In [ ]:
def synthetic_surface(heatmap: np.ndarray) -> SearchSurface:
    z = (heatmap - heatmap.mean()) / heatmap.std()
    size = (EDGE_CELLS, EDGE_CELLS)
    return SearchSurface(LatLon(50.0, 25.0), 0.0, z, 1.0, (0, 0), (0, 0), (DRONE_CELLS, DRONE_CELLS), size,
                         (EDGE_CELLS * MATCH_GSD_M, EDGE_CELLS * MATCH_GSD_M))

In [ ]:
centre = (HEATMAP_CELLS - 1) / 2
rows, cols = np.indices((HEATMAP_CELLS, HEATMAP_CELLS))
bump = synthetic_surface(np.exp(-((rows - centre) ** 2 + (cols - centre) ** 2) / (2 * BUMP_SIGMA_CELLS ** 2)))
line = synthetic_surface(np.exp(-(rows - centre) ** 2 / (2 * LINE_SIGMA_CELLS ** 2)))
config = CorrelatorConfig(search_scale=1.7, fov_w_deg=60.0)

In [ ]:
def axis_ratio(surface: SearchSurface, tau: float) -> float:
    features = gate_features(surface, config, (tau,))
    return float(np.exp(features[f"log_s1_t{tau:g}"] - features[f"log_s2_t{tau:g}"]))

In [ ]:
synthetic = pd.DataFrame({"tau": TAUS, "bump_s1_over_s2": [axis_ratio(bump, t) for t in TAUS],
                          "line_s1_over_s2": [axis_ratio(line, t) for t in TAUS]})
synthetic.round(3)

In [ ]:
assert (synthetic["bump_s1_over_s2"] < MAX_BUMP_AXIS_RATIO).all()
ridge = synthetic[synthetic["tau"] <= MAX_RIDGE_TAU]
assert (ridge["line_s1_over_s2"] > MIN_LINE_AXIS_RATIO).all()
assert synthetic["line_s1_over_s2"].is_monotonic_decreasing

# Data

In [ ]:
frames = [f for f in load_frames(FRAMES_DATASET) if json.loads((f.image_path.parent / "meta.json").read_text())["place"] == PLACE]
map_image = load_date(PLACE, MAP_DATE)
len(frames)

# Model

In [ ]:
model = load_segmenter(REPO_ROOT / SEG_CHECKPOINT, DEVICE, "lraspp")
features = SegmentationFeatures(model, model, map_image, "all", MATCH_GSD_M, MATCH_GSD_M, DEVICE)
correlator = PositionCorrelator(features, correlator_config(RUN_CONFIG))

# Run

In [ ]:
draws = benchmark_draws(correlator, frames, NOISE_M, N_DIRECTIONS, SEED)

In [ ]:
table = feature_table(draws, correlator.config, TAUS, 0.01)

In [ ]:
reference = pd.read_csv(experiment_dir("007-search-prior-and-gate").parent / REFERENCE_RECORDS)
reference = reference[reference["noise_m"] == NOISE_M].reset_index(drop=True)
keys = pd.DataFrame({"frame": [d.frame.name for d in draws], "bearing_deg": [d.bearing_deg for d in draws]})
joined = pd.concat([keys, table], axis=1).merge(reference, on=["frame"], suffixes=("", "_ref"))
joined = joined[np.isclose(joined["bearing_deg"], joined["bearing_deg_ref"])]
joined = joined[joined["eligible"] & joined["s1_m"].notna()]
len(joined)

In [ ]:
s1_diff = (np.exp(joined["log_s1_t0.25"]) - joined["s1_m"]).abs().max()
s2_diff = (np.exp(joined["log_s2_t0.25"]) - joined["s2_m"]).abs().max()
assert max(s1_diff, s2_diff) < MAX_FEATURE_DIFF
s1_diff, s2_diff